# Part 2: one hour, one traffic count

## Goal
Build a reproducible hourly target table and freeze chronological partitions. Missing observations stay unknown. This notebook checks structure, not model performance. Source: [Hogue (2019), UCI](https://doi.org/10.24432/C5X60B), CC BY 4.0.

## Setup
From the repository root run `python -m pip install -e .` and `python -m trafficpulse.data --download`. To execute this notebook, install `nbformat nbclient ipykernel nbconvert` and run `python -m jupyter nbconvert --execute --to notebook --inplace notebooks/02_hourly_quality.ipynb`. Notebook: Python 3.12+; the cleaning pipeline itself needs only the standard library.

In [1]:
from pathlib import Path
import sys, csv, hashlib, json
root = Path.cwd()
if not (root / "src").exists():
    root = root.parent
assert (root / "src/trafficpulse").exists(), "Open from repository root or notebooks directory"
sys.path.insert(0, str(root / "src"))
from trafficpulse.prepare import run
manifest = run()
print("Source SHA-256:", manifest["source_sha256"])

Source SHA-256: b99aeabcbd6cc86f642da3a79d90883425798f58abd3b3302da2fa19dda73768


## Steps
### 1. Confirm the hourly grid
Repeated timestamps retain one agreed traffic count, never a sum. Conflicting counts stop processing. Blank target cells represent absent hours. Measured zero is preserved and flagged. Quality flags describe the target hour and must not be used as predictive features.

In [2]:
print("Grid hours:", manifest["hourly_slots"])
print("Observed hours:", manifest["observed_hours"])
print("Missing hours:", manifest["missing_hours"])
assert manifest["hourly_slots"] == manifest["observed_hours"] + manifest["missing_hours"]

Grid hours: 52551
Observed hours: 40575
Missing hours: 11976


### 2. Inspect partition coverage
The boundaries were proposed before modeling: training before 2017, validation in 2017, and final test in 2018. We inspect only structural completeness across partitions, not target distributions. Coverage uses observed hours divided by all hourly slots within each partition.

In [3]:
print(f"{"Partition":<12} {"Slots":>8} {"Observed":>10} {"Missing":>9} {"Coverage":>10}")
for name, part in manifest["partitions"].items():
    print(f"{name:<12} {part["hourly_slots"]:>8,} {part["observed_hours"]:>10,} {part["missing_hours"]:>9,} {part["coverage"]:>9.2%}")

Partition       Slots   Observed   Missing   Coverage
train          37,239     25,329    11,910    68.02%
validation      8,760      8,713        47    99.46%
test            6,552      6,533        19    99.71%


Training coverage is about 68%, while validation and test coverage are above 99%. This is a limitation, not a reason to fill long gaps or choose splits based on model scores. Counts are conditional on the unresolved source-clock interpretation. A compact exact-value table is sufficient for these structural checks.

## Checks
Independently reconcile CSV timestamps and observed counts against the raw source. Confirm sorted unique hours, partition boundaries, and file fingerprints. No holdout outcome distribution is explored.

In [4]:
from datetime import datetime, timedelta
from trafficpulse.data import read_rows
raw = read_rows(root / "data/raw/source.zip")
raw_times = {r["date_time"] for r in raw}
with (root / "data/processed/hourly.csv").open(newline="") as f:
    grid = list(csv.DictReader(f))
stamps = [datetime.fromisoformat(r["date_time"]) for r in grid]
assert len(stamps) == len(set(stamps))
assert all(b-a == timedelta(hours=1) for a,b in zip(stamps, stamps[1:]))
assert {r["date_time"] for r in grid if r["observed"] == "1"} == raw_times
assert all(r["traffic_volume"] == "" for r in grid if r["observed"] == "0")
for name, output in manifest["outputs"].items():
    assert hashlib.sha256((root/output["path"]).read_bytes()).hexdigest() == output["sha256"]
for row in grid:
    expected = "train" if row["date_time"] < "2017-01-01" else "validation" if row["date_time"] < "2018-01-01" else "test"
    assert row["split"] == expected
print("PASS: grid, source reconciliation, missing encoding, split boundaries, and hashes")

PASS: grid, source reconciliation, missing encoding, split boundaries, and hashes


## Next Steps
Part 3 will explore training data and establish simple baselines. Weather and holiday fields remain in the raw source, excluded from this target-only table. Do not use observed-status, source-row-count or zero-volume flags as model inputs. Do not interpret volume as congestion.

**Exercise:** Why should a missing sensor reading stay blank rather than become zero?